# 02 — Model Eğitimi ve Karşılaştırma

Baseline, Linear Regression, Random Forest, Gradient Boosting modellerinin eğitimi ve MAE/RMSE karşılaştırması.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from train_model import train_all
from evaluate import print_comparison_table
from config import BEST_MODEL_FILE, METRICS_FILE

In [ ]:
# Tüm modelleri eğit ve metrikleri al
best_pipeline, metrics_df = train_all()

In [ ]:
print_comparison_table(metrics_df)

In [ ]:
# Görsel karşılaştırma
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

metrics_df.plot(
    x='model', y=['val_mae', 'test_mae'],
    kind='bar', ax=axes[0], color=colors[:2], edgecolor='white'
)
axes[0].set_title('MAE Karşılaştırması')
axes[0].set_ylabel('MAE')
axes[0].set_xticklabels(metrics_df['model'], rotation=25, ha='right')

metrics_df.plot(
    x='model', y=['val_rmse', 'test_rmse'],
    kind='bar', ax=axes[1], color=colors[2:], edgecolor='white'
)
axes[1].set_title('RMSE Karşılaştırması')
axes[1].set_ylabel('RMSE')
axes[1].set_xticklabels(metrics_df['model'], rotation=25, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# En iyi model özeti
best_entry = metrics_df[metrics_df['model'] != 'Baseline (rolling_mean_3_prev)'].nsmallest(1, 'test_mae')
print(f"En iyi model: {best_entry['model'].values[0]}")
print(f"Test MAE:    {best_entry['test_mae'].values[0]:.4f}")
print(f"Test RMSE:   {best_entry['test_rmse'].values[0]:.4f}")
print(f"\nModel dosyası: {BEST_MODEL_FILE}")

In [ ]:
# Random Forest feature önem sıralaması (model GradientBoosting ise de çalışır)
from feature_utils import build_preprocessor, load_splits_from_full, get_X_y
from config import CATEGORICAL_FEATURES, NUMERICAL_FEATURES

pipeline = joblib.load(BEST_MODEL_FILE)
model_step = pipeline.named_steps['model']

if hasattr(model_step, 'feature_importances_'):
    preprocessor = pipeline.named_steps['preprocessor']
    ohe_features = (
        preprocessor.named_transformers_['cat']
        .named_steps['ohe']
        .get_feature_names_out(CATEGORICAL_FEATURES)
        .tolist()
    )
    feature_names = ohe_features + NUMERICAL_FEATURES
    importances = model_step.feature_importances_

    fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
    fi_df = fi_df.nlargest(20, 'importance')

    fi_df.sort_values('importance').plot(
        x='feature', y='importance', kind='barh', figsize=(10, 8),
        color='steelblue', edgecolor='white', legend=False
    )
    plt.title('En Önemli 20 Feature')
    plt.tight_layout()
    plt.show()
else:
    print('Bu model feature importance desteklemiyor.')